# 使用 OpenAI 的网站内容摘要器（Playwright）

## 练习目标（理念）

做一个「抓网页 → 清洗正文 → 让模型写摘要」的小流水线：

1. **Playwright** 启动无头 Chromium，拿到 **JS 渲染后** 的完整 HTML（比单纯 `requests` 更能对付动态站）
2. **BeautifulSoup** 去掉 `script` / `nav` / `footer` 等噪音标签，抽出纯文本
3. **OpenAI Chat Completions** 生成简洁 Markdown 摘要

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | `async_playwright` + `page.content()` |
| 清洗 HTML | BeautifulSoup `decompose()` |
| system / user prompt | `messages_for(website)` |
| 展示结果 | `display(Markdown(summary))` |

## 怎么跑

1. 安装并配置好 Playwright 浏览器（`playwright install`）与 `.env` 里的 `OPENAI_API_KEY`
2. 从上到下运行；最后一格可把 URL 换成你想摘要的站点


In [ ]:
# ========== 导入：抓取 + 解析 + 调用模型所需工具 ==========

# 导入标准库 os：读环境变量（OPENAI_API_KEY）
import os
# load_dotenv：从 .env 加载密钥，避免把密钥写进代码
from dotenv import load_dotenv
# Playwright 异步 API：控制真实浏览器渲染页面
from playwright.async_api import async_playwright
# BeautifulSoup：解析 HTML、删除无用标签、提取文本
from bs4 import BeautifulSoup
# OpenAI 客户端：调用 Chat Completions 做摘要
from openai import OpenAI
# 在 Jupyter 里把摘要渲染成 Markdown
from IPython.display import Markdown, display


In [ ]:
# ========== 异步抓取：用 Playwright 取 JS 渲染后的正文文本 ==========

async def fetch_website_contents_js(url):
    # async with：进入时启动 Playwright，退出时自动清理
    async with async_playwright() as p:
        # headless=True：无头模式，不弹可见窗口
        browser = await p.chromium.launch(headless=True)
        # new_context：隔离的浏览器上下文；伪装常见桌面 Chrome UA + 视口
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
            viewport={"width": 1280, "height": 720}
        )
        # 在该上下文中打开新标签页
        page = await context.new_page()
        # goto：等待 DOMContentLoaded；超时 15s（毫秒）
        await page.goto(url, wait_until="domcontentloaded", timeout=15000)
        # 再等 3s：给 Cloudflare / 延迟脚本一点时间（原注释意图保留）
        await page.wait_for_timeout(3000)  # extra time for Cloudflare challenge
        # 取出渲染后的完整 HTML 字符串
        html = await page.content()
        # 关闭浏览器，释放资源
        await browser.close()

    # 用 html.parser 解析刚才拿到的 HTML
    soup = BeautifulSoup(html, "html.parser")
    # 删掉脚本、样式、导航、页脚、页头、图片、SVG 等噪音节点
    for tag in soup.find_all(["script", "style", "nav", "footer", "header", "img", "svg"]):
        tag.decompose()
    # 抽出可见文本：换行分隔，并 strip 每段空白
    return soup.get_text(separator="\n", strip=True)


### 初始化 OpenAI

加载 `.env`，检查 `OPENAI_API_KEY` 形态是否合理，再创建客户端。


In [ ]:
# ========== 加载并检查 OPENAI_API_KEY ==========

# override=True：让 .env 覆盖已有环境变量
load_dotenv(override=True)
# 读取密钥字符串
api_key = os.getenv('OPENAI_API_KEY')

# 分情况提示（文案保持英文，便于对照官方排错笔记本）

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
# ========== 创建默认 OpenAI 客户端（自动读环境变量里的密钥） ==========
openai = OpenAI()


### 提示词（Prompts）

`system_prompt` 定摘要风格；`user_prompt_prefix` 是用户消息的前缀，后面会拼上网页正文。
发给模型的英文指令不要翻译，否则摘要行为可能改变。


In [ ]:
# ========== Prompt：系统角色 + 用户前缀（英文原样保留） ==========

# system：告诉模型「你是摘要助手、请忽略导航、用 Markdown 直接回答」
system_prompt = """
You are an informative assistant that analyzes the contents of a website,
and provides a concise, clear, and factual summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

# user 前缀：后面会拼接 fetch 得到的网站正文
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


### 助手函数

把「网页正文」包装成 Chat Completions 需要的 `messages` 列表。


In [ ]:
# ========== 组装 messages：system +（前缀 + 网页正文） ==========

def messages_for(website):
    # 返回标准两段式对话：system 定规矩，user 携带页面文本
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [ ]:
# ========== 异步摘要：抓取 → 调模型 → 返回 content ==========

async def summarize_js(url):
    # 先异步抓取并清洗出纯文本
    website = await fetch_website_contents_js(url)
    # 再同步调用 Chat Completions（在 async 函数里直接调 SDK；模型 id 保持原样）
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages_for(website)
    )
    # 取出助手回复正文
    return response.choices[0].message.content


In [ ]:
# ========== 异步展示：摘要结果用 Markdown 显示 ==========

async def display_summary_js(url):
    # 复用 summarize_js 拿到字符串摘要
    summary = await summarize_js(url)
    # 在笔记本输出区渲染
    display(Markdown(summary))


## 总结网站

对目标 URL 跑完整流水线。把下面单元格里的地址换成你想分析的站点即可。


In [ ]:
# ========== 入口：对 openai.com 做一次 Playwright + GPT 摘要 ==========
# Jupyter 里可直接 await 顶层协程（IPython 事件循环已存在）
await display_summary_js("https://openai.com")
